In [ ]:
import glob, hashlib, json, os, time
SHA256, PARTS, TOTAL = "c3db5e1ac63b05efa67d6525826a8a792c0ce92b810e91ddd5ec0f42c73a175c", 6, 107011338717
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]
h, n, t0, rows = hashlib.sha256(), 0, time.time(), []
for i in range(PARTS):
    meta = json.load(open(find(f"flare-part-{i}.json")))
    assert meta["start"] == n, (i, meta, n)
    ph = hashlib.sha256()
    with open(find(f"flare-part-{i}"), "rb", buffering=0) as f:
        while True:
            b = f.read(64 << 20)
            if not b:
                break
            h.update(b); ph.update(b); n += len(b)
    rows.append({**meta, "rehash_ok": ph.hexdigest() == meta["sha256"]})
    print(rows[-1], f"{time.time() - t0:.0f} s", flush=True)
result = {"bytes": n, "bytes_ok": n == TOTAL, "sha256": h.hexdigest(), "sha256_ok": h.hexdigest() == SHA256,
          "parts": rows, "seconds": round(time.time() - t0, 1)}
json.dump(result, open("/kaggle/working/verify.json", "w"), indent=1)
print(json.dumps(result, indent=1))
assert result["bytes_ok"] and result["sha256_ok"] and all(r["rehash_ok"] for r in rows)
